<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [5]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [6]:
type(docs)

list

In [7]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [ ]:
import re

# liste de stop words basique, on enleve les mots qui apportent rien
STOP_WORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you',
    'your', 'yours', 'he', 'him', 'his', 'she', 'her', 'hers', 'it',
    'its', 'they', 'them', 'their', 'what', 'which', 'who', 'whom',
    'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were',
    'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can',
    'could', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because',
    'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
    'against', 'between', 'into', 'through', 'during', 'before',
    'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out',
    'on', 'off', 'over', 'under', 'again', 'then', 'once', 'so', 'not',
    'no', 'nor', 'very', 'just', 'also', 'both', 'each', 'few', 'more',
    'most', 'other', 'some', 'such', 'than', 'too', 'same', 'only'
])

def preprocess(text):
    # lowercase tout
    text = text.lower()
    # on garde seulement les lettres, on vire les chiffres et ponctuation
    text = re.sub(r'[^a-z\s]', '', text)
    # on split en tokens
    tokens = text.split()
    # on filtre les stop words et les mots trop courts (longueur < 2)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return tokens

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [ ]:
def build_vocab(docs):
    # on parcourt tous les docs et on collecte les mots uniques
    vocab = set()
    for doc in docs:
        tokens = preprocess(doc)
        vocab.update(tokens)
    # on trie pour avoir un ordre stable (important pr les indices du vecteur)
    vocab = sorted(list(vocab))
    return vocab

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
def compute_df(docs, vocab):
    # df[word] = nb de docs ou le mot apparait
    df = {word: 0 for word in vocab}
    for doc in docs:
        tokens = set(preprocess(doc))  # set() pour compter chaque mot 1 seule fois par doc
        for token in tokens:
            if token in df:
                df[token] += 1
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
import math

def compute_idf(df, N):
    # formule smoothed IDF = log((N+1) / (df+1)) + 1
    # le +1 dans le log c'est le lissage pr eviter division par 0
    # le +1 en dehors du log c'est pr que meme si log vaut 0 on garde qqchose
    idf = {}
    for word, freq in df.items():
        idf[word] = math.log((N + 1) / (freq + 1)) + 1
    return idf

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [ ]:
def compute_tf(doc, vocab):
    # tf[word] = count(word in doc) / total words in doc
    tokens = preprocess(doc)
    total = len(tokens)
    tf = {word: 0 for word in vocab}
    if total == 0:
        return tf  # doc vide, on retourne des zeros
    for token in tokens:
        if token in tf:
            tf[token] += 1
    # normalisation par le nb total de mots
    for word in tf:
        tf[word] = tf[word] / total
    return tf

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [ ]:
import numpy as np

def build_tfidf(docs):
    # step 1: construire le vocab
    vocab = build_vocab(docs)
    N = len(docs)

    # step 2: calculer df puis idf
    df = compute_df(docs, vocab)
    idf = compute_idf(df, N)

    # step 3: construire la matrice tfidf (nb_docs x vocab_size)
    tfidf_matrix = []
    for doc in docs:
        tf = compute_tf(doc, vocab)
        # vecteur tfidf = tf * idf pr chaque mot du vocab
        vec = np.array([tf[word] * idf[word] for word in vocab])
        tfidf_matrix.append(vec)

    tfidf_matrix = np.array(tfidf_matrix)
    return tfidf_matrix, vocab

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [ ]:
def cosine(a, b):
    # formule: cos(theta) = (A.B) / (||A|| * ||B||)
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    # si un des deux vecteurs est nul on retourne 0 (pas de similarite)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [8]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [ ]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)